# Chapter 7: Dynamic Creative Optimization

This notebook contains the implementation details for the Dynamic Creative Optimization (DCO) chapter.

## Code Listing 7.1: Vectorized Thompson Sampling

In [ ]:
import torch

def select_creative_thompson_sampling(mu, sigma_sq, available_mask):
    """
    Vectorized Thompson Sampling selection.
    
    Args:
        mu (Tensor): Mean estimated reward for each creative [batch_size, num_creatives]
        sigma_sq (Tensor): Uncertainty (variance) for each creative [batch_size, num_creatives]
        available_mask (Tensor): Boolean mask (1=valid, 0=ineligible) [batch_size, num_creatives]
        
    Returns:
        selected_indices (Tensor): Index of selected creative per request
    """
    batch_size, num_creatives = mu.shape
    
    # 1. Sample from posterior N(mu, sigma)
    # std_dev = sqrt(variance)
    std_dev = torch.sqrt(sigma_sq)
    
    # eps ~ N(0, 1)
    eps = torch.randn_like(mu)
    
    # sampled_reward = mu + eps * std
    sampled_rewards = mu + (eps * std_dev)
    
    # 2. Mask ineligible creatives (set reward to -infinity)
    # This prevents selecting creatives that failed policy filters or are out of stock
    NEG_INF = -1e9
    masked_rewards = sampled_rewards.masked_fill(~available_mask, NEG_INF)
    
    # 3. Greedy selection on sampled values
    selected_indices = torch.argmax(masked_rewards, dim=1)
    
    return selected_indices

# Test the function
batch_size = 2
num_creatives = 5

# Mock data
mu = torch.rand(batch_size, num_creatives)
sigma_sq = torch.rand(batch_size, num_creatives) * 0.1
available_mask = torch.ones(batch_size, num_creatives, dtype=torch.bool)
# Make last creative unavailable
available_mask[:, -1] = False

selected = select_creative_thompson_sampling(mu, sigma_sq, available_mask)
print(f"Selected indices: {selected}")

## Code Listing 7.2: Constraint-Aware Re-Ranking

In [ ]:
def constraint_aware_reranking(candidates, user_history, budget_status, lambda_fatigue=0.5):
    """
    Adjusts creative scores based on fatigue and diversity constraints.
    
    Args:
        candidates (list): List of dicts {'id', 'score', 'embedding'}
        user_history (list): IDs of creatives seen by user in last 24h
        budget_status (dict): Maps creative_id -> is_over_budget (bool)
    
    Returns:
        final_candidate (dict): The best candidate satisfying constraints
    """
    best_score = -1.0
    best_candidate = None
    
    for cand in candidates:
        original_score = cand['score']
        cid = cand['id']
        
        # 1. Hard Filter: Budget Pacing
        if budget_status.get(cid, False):
            continue  # Skip if budget exhausted
            
        # 2. Soft Penalty: Fatigue
        # Count impressions in user history
        impressions = user_history.count(cid)
        # Apply exponential decay: score * e^(-lambda * count)
        fatigue_multiplier = 2.718 ** (-lambda_fatigue * impressions)
        
        # 3. Diversity Penalty (Simplified)
        # If user saw this exact creative recently, apply massive penalty
        # (In production, use embedding similarity)
        recent_penalty = 0.0
        if impressions > 0 and user_history[-1] == cid:
             recent_penalty = 0.5  # Heavy penalty for back-to-back repeats
             
        final_score = (original_score * fatigue_multiplier) - recent_penalty
        
        if final_score > best_score:
            best_score = final_score
            best_candidate = cand
            
    return best_candidate

# Test
candidates = [
    {'id': 'c1', 'score': 0.9},
    {'id': 'c2', 'score': 0.85},
    {'id': 'c3', 'score': 0.8}
]
user_history = ['c1', 'c1'] # User saw c1 twice
budget_status = {'c2': True} # c2 is over budget

best = constraint_aware_reranking(candidates, user_history, budget_status)
print(f"Best candidate: {best['id'] if best else 'None'}")

## Code Listing 7.3: Multi-Task Image-Level Reward Model

In [ ]:
import torch
import torch.nn as nn

class MultiTaskImageRewardModel(nn.Module):
    """
    Evaluates a rendered creative using multiple parallel heads:
    1. pCTR (Performance)
    2. Brand Consistency Score
    3. Policy Risk Score
    """
    def __init__(self, visual_dim=768, text_dim=768, hidden_dim=256):
        super().__init__()
        
        # Shared Fusion Trunk
        # Concatenates visual embedding (CLIP/SigLIP) with copy embedding
        self.fusion_layer = nn.Sequential(
            nn.Linear(visual_dim + text_dim, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU()
        )
        
        # Head 1: Performance Reward (Proxy for CTR)
        self.ctr_head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid() # Probability [0, 1]
        )
        
        # Head 2: Brand Consistency (Similarity to brand guidelines)
        self.brand_head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid() # Score [0, 1]
        )
        
        # Head 3: Policy Risk (Probability of violation)
        self.risk_head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid() # Probability [0, 1]
        )

    def forward(self, visual_emb, text_emb):
        # inputs: [batch_size, 768]
        combined = torch.cat([visual_emb, text_emb], dim=1)
        shared_repr = self.fusion_layer(combined)
        
        return {
            "p_ctr": self.ctr_head(shared_repr),
            "brand_score": self.brand_head(shared_repr),
            "risk_score": self.risk_head(shared_repr)
        }

# Test model
model = MultiTaskImageRewardModel()
visual_emb = torch.randn(2, 768)
text_emb = torch.randn(2, 768)
outputs = model(visual_emb, text_emb)
print(outputs)